In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Variables

In [1]:
# Gravity
g = 9.8

# Arm 1
l1 = 0.5
m1 = 1
theta1_0 = 0
v1_0 = 2

# Arm 2
l2 = l1
m2 = 1
theta2_0 = 0
v2_0 = 1

# Combined Mass
M = m1+m2

# Length of simulation
t1 = 10

# Step size
h = 0.0005

## RK6 Function

In [22]:
#Define the function
def RK6(t1, theta1_0, theta2_0, v1_0, v2_0, g, h,count):
    
    #Initialise the arrays to be used
    # t is an array containing each of the timepoints that we will step forward to
    t = np.arange(0,t1+h,h)
    # n is the number of timesteps in t
    n = np.shape(t)[0]
    # theta starts as an empty array, but we will fill in the values we calculate in the loop, below
    theta1 = np.zeros(n)
    theta2 = np.zeros(n)
    #same for velocities
    v1 = np.zeros(n)
    v2 = np.zeros(n)
    
    
    
    #Set the initial value of x and v
    theta1[0] = theta1_0
    v1[0] = v1_0
    theta2[0] = theta2_0
    v2[0] = v2_0
    
    #define F(theta) for coupled equation 1
    def F1(theta1, theta2, v1, v2):
        
        thetaDiff = theta1-theta2
        a = m1 + m2*(np.sin(thetaDiff))**2
        
        return (-np.sin(thetaDiff)*(m2*l1*v1**2*np.cos(thetaDiff)+m2*l2*v2**2)-g*(M*np.sin(theta1)-m2*np.sin(theta2)*np.cos(thetaDiff)))/(l1*a)
    
    #Define G(v) for coupled equation 2
    def G1(v1):
        
        return v1
    
        #define F(theta) for coupled equation 1
    def F2(theta1, theta2, v1, v2):
        
        thetaDiff = theta1-theta2
        a = m1 + m2*(np.sin(thetaDiff))**2
        
        return (np.sin(thetaDiff)*(M*l1*v1**2+m2*l2*v2**2*np.cos(thetaDiff))+g*(M*np.sin(theta1)*np.cos(thetaDiff)-M*np.sin(theta2)))/(l2*a)
    
    #Define G(v) for coupled equation 2
    def G2(v2):
        
        return v2
    
    
    #Loop over the time values and calculate the derivatives
    for i in range(1,n): 
        
        # Calculate the Runge - Kutta intermediate values
        f1 = h * F1(theta1[i-1], theta2[i-1], v1[i-1], v2[i-1])
        f2 = h * F1(theta1[i-1] + h/4, theta2[i-1] + h/4, v1[i-1] + h/4, v2[i-1] + h/4)
        f3 = h * F1(theta1[i-1] + h/4, theta2[i-1] + h/4, v1[i-1] + h/4, v2[i-1] + h/4)
        f4 = h * F1(theta1[i-1] + h/2, theta2[i-1] + h/2, v1[i-1] + h/2, v2[i-1] + h/2)
        f5 = h * F1(theta1[i-1] + 3*h/4, theta2[i-1] + 3*h/4, v1[i-1] + 3*h/4, v2[i-1] + 3*h/4)
        f6 = h * F1(theta1[i-1] + h, theta2[i-1] + h, v1[i-1] + h, v2[i-1] + h)
        
        g1 = h * G1(v1[i-1])
        g2 = h * G1(v1[i-1] + h/4)
        g3 = h * G1(v1[i-1] + h/4)
        g4 = h * G1(v1[i-1] + h/2)
        g5 = h * G1(v1[i-1] + 3*h/4)
        g6 = h * G1(v1[i-1] + h)
        
        e1 = h * F2(theta1[i-1], theta2[i-1], v1[i-1], v2[i-1])
        e2 = h * F2(theta1[i-1] + e1/4, theta2[i-1] + e1/4, v1[i-1] + e1/4, v2[i-1] + e1/4)
        e3 = h * F2(theta1[i-1] + e1/4, theta2[i-1] + e1/4, v1[i-1] + e1/4, v2[i-1] + e1/4)
        e4 = h * F2(theta1[i-1] + e2/2, theta2[i-1] + e2/2, v1[i-1] + e2/2, v2[i-1] + e2/2)
        e5 = h * F2(theta1[i-1] + 3*e3/4, theta2[i-1] + 3*e3/4, v1[i-1] + 3*e3/4, v2[i-1] + 3*e3/4)
        e6 = h * F2(theta1[i-1] + e4, theta2[i-1] + e4, v1[i-1] + e4, v2[i-1] + e4)
        
        j1 = h * G2(v2[i-1])
        j2 = h * G2(v2[i-1] + j1/4)
        j3 = h * G2(v2[i-1] + j1/8 + j2/8)
        j4 = h * G2(v2[i-1] - j2/2 + j3)
        j5 = h * G2(v2[i-1] + 3*j1/16 + 9*j4/16)
        j6 = h * G2(v2[i-1] - 3*j1/7 + 2*j2/7 + 12*j3/7 - 12*j4/7 + 8*j5/7)

        
        theta1[i] = theta1[i-1] + (1/6 * g1 + 1/3 * g2 + 1/3 * g3 + 1/6 * g4) + (1/6 * g5 + 1/6 * g6)
        v1[i] = v1[i-1] + (1/6 * f1 + 1/3 * f2 + 1/3 * f3 + 1/6 * f4) + (1/6 * f5 + 1/6 * f6)

        theta2[i] = theta2[i-1] + (1/6 * j1 + 1/3 * j2 + 1/3 * j3 + 1/6 * j4) + (1/6 * j5 + 1/6 * j6)
        v2[i] = v2[i-1] + (1/6 * e1 + 1/3 * e2 + 1/3 * e3 + 1/6 * e4) + (1/6 * e5 + 1/6 * e6)

        
        if theta1[i] >= 2*np.pi:
            theta1[i] = theta1[i]-2*np.pi
            
        count = 0
        if theta1[i] == 0:
            count +=1
        
    return(t,theta1,theta2,v1,v2,count) 

In [23]:
#record start time so we can measure run-time of program
start_time = time.time()

t, theta1,theta2, v1,v2 = RK6(t1, theta1_0, theta2_0, v1_0, v2_0, g, h,count)
print(theta1, v1)

#print program run-time 
print("--- %s seconds ---" % (time.time() - start_time))

NameError: name 'count' is not defined

## Plot of Theta 1 against time

In [ ]:
plt.plot(t,theta1)
plt.title('Theta 1 against time', weight='bold')
plt.xlabel('Time')
plt.ylabel('Theta 1 (rads)')
plt.show()

## Lissajou Figure

In [ ]:
plt.plot(theta1, theta2)
plt.xlabel('Theta1 (rads)')
plt.ylabel('Theta2 (rads)')
plt.title('Lissajou Figure', weight='bold')
plt.show()

## Visualisation of Pendulum

In [ ]:
from PIL import Image
import time, os

In [15]:
start_time = time.time()

t, theta1,theta2, v1,v2 = RK6(t1, theta1_0, theta2_0, v1_0, v2_0, g, 0.0005)

x1 = l1*np.sin(theta1)
y1 = -l1*np.cos(theta1)

x2 = x1 + l2*np.sin(theta2)
y2 = y1 - l2*np.cos(theta2)
print(theta1)
frames = []
for i in range(len(x1)):
    if i%50 == 0:
        fig=plt.figure()
        plt.plot((0, x1[i]), (0, y1[i]))
        plt.plot((x1[i], x2[i]), (y1[i], y2[i]))
        plt.xlim(-1, 1)
        plt.ylim(-0.6, 0.6)
    
        canvas = plt.get_current_fig_manager().canvas
        canvas.draw()
        im = Image.frombytes('RGB', canvas.get_width_height(), canvas.tostring_rgb())
        frames.append(im)
            
    plt.close(fig)
    #plt.show(fig)

frames[0].save("filename.gif", format='GIF', append_images=frames[1:], save_all=True, duration=100, loop=0)

print("--- %s seconds ---" % (time.time() - start_time))

[0.         0.00133347 0.00266694 ... 0.33115096 0.33217839 0.33320166]
--- 37.25631618499756 seconds ---


### References

For Runga-Kutte: https://www.physics.umd.edu/hep/drew/pendulum2.html